# 01 — Reproducible Banking77 Data Preparation

Run this notebook from top to bottom in VS Code or Google Colab. A CPU is sufficient;
the first run needs internet access to download Banking77. In Colab, Google Drive is mounted automatically.

We keep one validation set and prepare three training draws (42, 43, 44) for each
setting. The sampling order reproduces the original notebook's validation and seed-42
training sets. Existing files from `01_data_preparation.ipynb` are read only for comparison.

Outputs go into `data/reproducible/` under the project directory. Identical saved
artifacts are reused; a conflicting artifact stops the run instead of being overwritten.

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

required_packages = {"datasets": "5.1.0", "huggingface_hub": "2.2.0"}
packages_to_install = []
loaded_packages_to_update = []
for package, version in required_packages.items():
    try:
        installed_version = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed_version = None
    if installed_version != version:
        packages_to_install.append(f"{package}=={version}")
        if package in sys.modules:
            loaded_packages_to_update.append(package)

if packages_to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages_to_install])
    if loaded_packages_to_update:
        raise RuntimeError("Dependencies changed. Restart the kernel, then run all cells.")
    print("Required dataset packages installed.")
else:
    print("Required dataset packages are available.")

## 1. Paths and experiment settings

Locally, the project is detected from the repository root or its `notebooks/` folder.
Colab uses `MyDrive/CS682_Neural_Project`. Change `PROJECT_ROOT_OVERRIDE` if needed.
The existing sibling `project/data/` folder is detected automatically for comparison.
All new data is saved separately under this repository's `data/reproducible/`.

The dataset is pinned to an immutable Hugging Face commit. Validation uses seed 42
once; only the training draws change. The 15 rare classes are the smallest classes
in the original training split, matching the existing notebook.

In [ ]:
import hashlib
import json
import os
import platform
import random
from collections import Counter, defaultdict
from pathlib import Path

PROJECT_ROOT_OVERRIDE = None
DATASET_ID = "PolyAI/banking77"
DATASET_REVISION = "1846984189f2bc0ac51c6fc3a600fa9e1714e0c6"
VALIDATION_SEED = 42
TRAINING_SEEDS = (42, 43, 44)
VALIDATION_PER_CLASS = 10
LOW_RESOURCE_PER_CLASS = 10
RARE_CLASS_COUNT = 15
RARE_PER_CLASS = 5
COMMON_PER_CLASS = 20

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive")

if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/CS682_Neural_Project")
else:
    PROJECT_ROOT = next(
        (path for path in (Path.cwd(), *Path.cwd().parents)
         if (path / "notebooks").is_dir() and (path / "README.md").is_file()),
        None,
    )
    if PROJECT_ROOT is None:
        raise RuntimeError("Set PROJECT_ROOT_OVERRIDE to your project directory, then rerun.")

OUTPUT_ROOT = Path(os.environ.get("CS682_OUTPUT_ROOT", PROJECT_ROOT / "data" / "reproducible")).expanduser().resolve()
legacy_candidates = (PROJECT_ROOT / "data", PROJECT_ROOT.parent / "data")
LEGACY_DATA_ROOT = next(
    (path for path in legacy_candidates if (path / "processed" / "validation_data").is_dir()),
    None,
)
if os.environ.get("CS682_LEGACY_DATA_ROOT"):
    LEGACY_DATA_ROOT = Path(os.environ["CS682_LEGACY_DATA_ROOT"]).expanduser().resolve()

# Keep download caches with the output data, including in Colab sessions.
os.environ["HF_HOME"] = str(OUTPUT_ROOT / "_cache")
os.environ["HF_HUB_CACHE"] = str(OUTPUT_ROOT / "_cache" / "hub")
os.environ["HF_DATASETS_CACHE"] = str(OUTPUT_ROOT / "_cache" / "datasets")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_ROOT)
print("New outputs:", OUTPUT_ROOT)
print("Existing data:", LEGACY_DATA_ROOT or "Not found; reproduce from the pinned source")

## 2. Load and identify the official data

`source_id` identifies a row in the original split (for example, `train:125`).
The original text, labels, and row order are preserved. On subsequent runs, the
saved raw data is loaded locally without downloading the dataset again.

In [ ]:
import pandas as pd
from IPython.display import display
from datasets import DatasetDict, load_dataset, load_from_disk


def require(condition, message):
    if not condition:
        raise ValueError(message)


def dataset_digest(data):
    digest = hashlib.sha256()
    digest.update(json.dumps(data.features.to_dict(), sort_keys=True).encode("utf-8"))
    for row in data:
        digest.update(json.dumps(row, sort_keys=True, ensure_ascii=False).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()


raw_path = OUTPUT_ROOT / "raw" / "banking77"
source_path = OUTPUT_ROOT / "raw" / "source.json"
if raw_path.exists():
    require(source_path.is_file(), "Saved raw data is missing source.json; use a fresh output directory.")
    saved_source = json.loads(source_path.read_text())
    require(saved_source["dataset_id"] == DATASET_ID, "Saved dataset ID differs from the configuration.")
    require(saved_source["revision"] == DATASET_REVISION, "Saved dataset revision differs; use a fresh output directory.")
    raw_data = load_from_disk(str(raw_path))
else:
    downloaded = load_dataset(
        DATASET_ID, revision=DATASET_REVISION,
        cache_dir=str(OUTPUT_ROOT / "_cache" / "datasets"), token=False,
    )
    raw_data = DatasetDict({
        split: data.add_column("source_row", list(range(len(data)))).add_column(
            "source_id", [f"{split}:{index}" for index in range(len(data))]
        )
        for split, data in downloaded.items()
    })

require(set(raw_data) == {"train", "test"}, "Expected the official train and test splits.")
require(len(raw_data["train"]) == 10003 and len(raw_data["test"]) == 3080, "Unexpected official dataset sizes.")
class_names = raw_data["train"].features["label"].names
require(len(class_names) == 77, "Expected 77 intent classes.")
require(raw_data["test"].features["label"].names == class_names, "Train/test label mappings differ.")
for split, data in raw_data.items():
    require(data["source_id"] == [f"{split}:{index}" for index in range(len(data))], "Invalid source IDs.")
    require(data["source_row"] == list(range(len(data))), "Invalid original row indices.")

source_manifest = {
    "dataset_id": DATASET_ID,
    "revision": DATASET_REVISION,
    "class_names": class_names,
    "rows": {split: len(data) for split, data in raw_data.items()},
    "sha256": {split: dataset_digest(data) for split, data in raw_data.items()},
}
if source_path.exists():
    require(json.loads(source_path.read_text()) == source_manifest, "Saved raw data does not match its source manifest.")

display(pd.DataFrame([{"split": split, "rows": len(data), "classes": len(set(data["label"]))}
                      for split, data in raw_data.items()]))

## 3. Build the fixed validation split and three training draws

Class encounter order and a separate Python random generator for each sampling
operation preserve the original seed-42 selections. Every training draw comes from
the post-validation pool. Overlap between draws or between the two settings is allowed.

In [ ]:
def group_indices(data):
    groups = defaultdict(list)
    for index, label in enumerate(data["label"]):
        groups[label].append(index)
    return groups


def sample_indices(data, per_class, seed):
    rng = random.Random(seed)
    selected = []
    for label, indices in group_indices(data).items():
        require(len(indices) >= per_class[label], f"Not enough training examples for label {label}.")
        selected.extend(rng.sample(indices, per_class[label]))
    return selected


original_train = raw_data["train"]
all_labels = list(range(len(class_names)))
validation_counts = {label: VALIDATION_PER_CLASS for label in all_labels}
validation_indices = sample_indices(original_train, validation_counts, VALIDATION_SEED)
validation_index_set = set(validation_indices)
remaining_indices = [
    index for indices in group_indices(original_train).values()
    for index in indices if index not in validation_index_set
]
validation_data = original_train.select(validation_indices)
remaining_train_data = original_train.select(remaining_indices)

original_counts = Counter(original_train["label"])
# Stable sorting also preserves the original tie order for equally frequent labels.
rare_classes = [label for label, count in sorted(original_counts.items(), key=lambda item: item[1])[:RARE_CLASS_COUNT]]
rare_class_set = set(rare_classes)
setting_counts = {
    "low_resource": {label: LOW_RESOURCE_PER_CLASS for label in all_labels},
    "imbalanced": {label: RARE_PER_CLASS if label in rare_class_set else COMMON_PER_CLASS for label in all_labels},
}
training_data = {
    (setting, seed): remaining_train_data.select(sample_indices(remaining_train_data, counts, seed))
    for setting, counts in setting_counts.items() for seed in TRAINING_SEEDS
}

display(pd.DataFrame([{"label": label, "intent": class_names[label], "original_train_rows": original_counts[label]}
                      for label in rare_classes]))

## 4. Validate the splits and compare with the existing data

Checks cover class counts, blank text, repeated text, source row separation,
sampling reproducibility, and agreement with the saved seed-42 datasets when present.
Text checks normalize case and whitespace. Source-data train/test text overlap is
reported without removing or replacing any official test examples.

The remaining pool is saved for provenance and future sampling. Each experiment must
use only its selected training subset for classifier training, generation prompts,
EDA, and training the F2 classifier. Validation is additional labeled supervision
used for model and threshold selection; test data is reserved for final evaluation.

In [ ]:
def normalize_text(text):
    return " ".join(text.casefold().split())


def normalized_texts(data):
    return {normalize_text(text) for text in data["text"]}


def validate_dataset(data, expected_counts, name):
    require(Counter(data["label"]) == Counter(expected_counts), f"{name}: incorrect per-class counts.")
    require(len(set(data["source_id"])) == len(data), f"{name}: repeated source rows.")
    require(all(isinstance(text, str) and text.strip() for text in data["text"]), f"{name}: blank or invalid text.")
    require(len(normalized_texts(data)) == len(data), f"{name}: repeated normalized text.")
    require(data.features["label"].names == class_names, f"{name}: label mapping changed.")


validate_dataset(validation_data, validation_counts, "validation")
require(len(remaining_train_data) == 9233, "Unexpected post-validation pool size.")
validation_ids = set(validation_data["source_id"])
pool_ids = set(remaining_train_data["source_id"])
require(not validation_ids & pool_ids, "Validation overlaps the training pool.")
require(validation_ids | pool_ids == set(original_train["source_id"]), "Training pool and validation do not partition original train.")
require(not normalized_texts(validation_data) & normalized_texts(remaining_train_data), "Validation text overlaps the training pool.")

summary_rows = []
for (setting, seed), data in training_data.items():
    name = f"{setting}/seed_{seed}"
    validate_dataset(data, setting_counts[setting], name)
    require(set(data["source_id"]) <= pool_ids, f"{name}: examples outside the training pool.")
    require(not set(data["source_id"]) & validation_ids, f"{name}: validation source leakage.")
    require(not normalized_texts(data) & normalized_texts(validation_data), f"{name}: validation text leakage.")
    repeated = remaining_train_data.select(sample_indices(remaining_train_data, setting_counts[setting], seed))
    require(data["source_id"] == repeated["source_id"], f"{name}: sampling is not repeatable.")
    summary_rows.append({"setting": setting, "seed": seed, "rows": len(data), "classes": len(set(data["label"]))})

for setting in setting_counts:
    draws = {tuple(training_data[(setting, seed)]["source_id"]) for seed in TRAINING_SEEDS}
    require(len(draws) == len(TRAINING_SEEDS), f"{setting}: two training draws are identical.")

legacy_checks = {}
legacy_comparisons = {
    "validation_data": validation_data,
    "remaining_train_data": remaining_train_data,
    "low_resource_data": training_data[("low_resource", 42)],
    "imbalanced_data": training_data[("imbalanced", 42)],
}
for name, current in legacy_comparisons.items():
    path = LEGACY_DATA_ROOT / "processed" / name if LEGACY_DATA_ROOT else None
    if path is None or not path.exists():
        legacy_checks[name] = "not available"
        continue
    previous = load_from_disk(str(path))
    require(previous["text"] == current["text"] and previous["label"] == current["label"],
            f"{name}: does not reproduce the saved original; stop before saving new artifacts.")
    require(previous.features["label"].names == class_names, f"{name}: legacy label mapping differs.")
    legacy_checks[name] = "exact ordered match"

source_overlap = {
    "normalization": "casefold and collapse whitespace",
    "official_train_test_shared_texts": len(normalized_texts(original_train) & normalized_texts(raw_data["test"])),
    "validation_test_shared_texts": len(normalized_texts(validation_data) & normalized_texts(raw_data["test"])),
    "training_test_shared_texts": {
        f"{setting}/seed_{seed}": len(normalized_texts(data) & normalized_texts(raw_data["test"]))
        for (setting, seed), data in training_data.items()
    },
}
display(pd.DataFrame(summary_rows))
print("Legacy comparisons:", json.dumps(legacy_checks, indent=2))
print("Source-data text overlap audit:", json.dumps(source_overlap, indent=2))
print("All preparation checks passed.")

## 5. Save datasets, source IDs, configuration, and checks

The experiment manifest records the pinned source, fixed rare classes, seeds,
per-class counts, ordered source IDs, and content hashes. All existing datasets and
JSON files are checked before writing; incompatible files require a new output folder.
Package versions are recorded separately for each distinct runtime environment.

In [ ]:
prepared_data = {
    "validation_data": validation_data,
    "remaining_train_data": remaining_train_data,
    **{f"{setting}/seed_{seed}": data for (setting, seed), data in training_data.items()},
}
dataset_artifacts = {raw_path / split: data for split, data in raw_data.items()}
dataset_artifacts.update({OUTPUT_ROOT / "processed" / name: data for name, data in prepared_data.items()})

config = {
    "schema_version": 1,
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "validation_seed": VALIDATION_SEED,
    "training_seeds": list(TRAINING_SEEDS),
    "validation_per_class": VALIDATION_PER_CLASS,
    "low_resource_per_class": LOW_RESOURCE_PER_CLASS,
    "rare_class_count": RARE_CLASS_COUNT,
    "rare_per_class": RARE_PER_CLASS,
    "common_per_class": COMMON_PER_CLASS,
    "rare_class_policy": "smallest original-training classes; stable source-encounter tie order",
    "rare_class_ids": rare_classes,
    "class_names": class_names,
    "sampling_algorithm": "Python random.Random(seed).sample; original class encounter order; reset per setting",
    "source_id_format": "official_split:zero_based_original_row",
}
manifest = {
    "config": config,
    "datasets": {
        str(path.relative_to(OUTPUT_ROOT)): {
            "rows": len(data),
            "per_class": {str(label): count for label, count in sorted(Counter(data["label"]).items())},
            "source_ids": list(data["source_id"]),
            "sha256": dataset_digest(data),
        }
        for path, data in dataset_artifacts.items()
    },
}
runtime = {
    "python": platform.python_version(),
    "packages": {package: metadata.version(package) for package in ("datasets", "huggingface_hub", "pyarrow", "pandas", "numpy")},
}
runtime_key = hashlib.sha256(json.dumps(runtime, sort_keys=True).encode()).hexdigest()[:12]
json_artifacts = {
    source_path: source_manifest,
    OUTPUT_ROOT / "manifests" / "experiment_manifest.json": manifest,
    OUTPUT_ROOT / "manifests" / "rare_classes.json": {
        "policy": config["rare_class_policy"],
        "classes": [{"id": label, "name": class_names[label], "original_train_count": original_counts[label]}
                    for label in rare_classes],
    },
    OUTPUT_ROOT / "manifests" / "data_checks.json": {"status": "passed", "source_overlap": source_overlap},
    OUTPUT_ROOT / "manifests" / f"environment_{runtime_key}.json": runtime,
}

# Preflight every artifact before saving, so reruns never overwrite incompatible data.
for path, data in dataset_artifacts.items():
    if path.exists():
        existing = load_from_disk(str(path))
        require(dataset_digest(existing) == dataset_digest(data), f"Existing data differs: {path}. Use a new OUTPUT_ROOT.")
for path, value in json_artifacts.items():
    if path.exists():
        require(json.loads(path.read_text()) == value, f"Existing metadata differs: {path}. Use a new OUTPUT_ROOT.")

if not raw_path.exists():
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_data.save_to_disk(str(raw_path))
for name, data in prepared_data.items():
    path = OUTPUT_ROOT / "processed" / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        data.save_to_disk(str(path))
for path, value in json_artifacts.items():
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

for path, data in dataset_artifacts.items():
    require(dataset_digest(load_from_disk(str(path))) == dataset_digest(data), f"Save/load verification failed: {path}")

print("Saved and reloaded the official train/test data, shared validation/pool, and six training datasets.")
print("Experiment manifest:", OUTPUT_ROOT / "manifests" / "experiment_manifest.json")

## 6. Load the inputs for the next notebook

Use the same seed for the real subset and all augmentation conditions within a run.
Select `low_resource` or `imbalanced` and one of 42, 43, 44 below. `source_id` and
`source_row` are provenance columns; only `text` is the classifier input.
The official test set is stored at `raw/banking77/test` and remains reserved for
final evaluation. Saving and auditing it here does not tune any sampling or model choice.

In [ ]:
SETTING = "low_resource"
SEED = 42

train_data = load_from_disk(str(OUTPUT_ROOT / "processed" / SETTING / f"seed_{SEED}"))
val_data = load_from_disk(str(OUTPUT_ROOT / "processed" / "validation_data"))
experiment = json.loads((OUTPUT_ROOT / "manifests" / "experiment_manifest.json").read_text())
rare_class_ids = experiment["config"]["rare_class_ids"]

print(f"Ready for B1: {SETTING}, seed {SEED}, {len(train_data)} training rows, {len(val_data)} validation rows.")
display(pd.DataFrame(train_data.select(range(5)).to_dict()))

References: [Banking77 source](https://huggingface.co/datasets/PolyAI/banking77),
[Hugging Face dataset loading and revisions](https://huggingface.co/docs/datasets/loading),
and [dataset selection and persistence](https://huggingface.co/docs/datasets/package_reference/main_classes).